In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install torch numpy pandas matplotlib tqdm

In [ ]:
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

DATASET_DIR = Path("/content/drive/MyDrive/carewave_dataset")
DATA_DIR = DATASET_DIR / "processed" / "dataset"
MODEL_DIR = DATASET_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "csi_pose_best_bilstm_attention.pth"

BATCH_SIZE = 256
EPOCHS = 120
PATIENCE = 18
LR = 1e-3
HIDDEN_DIM = 256
NUM_LAYERS = 2
SMOOTHNESS_WEIGHT = 0.01
AUX_WEIGHT = 0.2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


class CSIPoseBiLSTMAttention(nn.Module):
    def __init__(self, input_dim=156, hidden_dim=256, num_layers=2, output_dim=132):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
        )

        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2 if num_layers > 1 else 0.0,
            bidirectional=True,
        )

        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_dim * 2,
            num_heads=8,
            dropout=0.1,
            batch_first=True,
        )

        self.norm1 = nn.LayerNorm(hidden_dim * 2)
        self.norm2 = nn.LayerNorm(hidden_dim * 2)

        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
        )

        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, output_dim),
        )

    def forward(self, x):
        x = self.input_proj(x)
        out, _ = self.lstm(x)

        attn_out, _ = self.attn(out, out, out)
        out = self.norm1(out + attn_out)

        ffn_out = self.ffn(out)
        out = self.norm2(out + ffn_out)

        return self.head(out)


def weighted_mse(pred, target, visibility):
    weights = visibility.clamp(0.05, 1.0)
    error = (pred - target).pow(2) * weights
    return error.sum() / weights.sum().clamp_min(1.0)


def temporal_smoothness_loss(pred_raw):
    return (pred_raw[:, 1:] - pred_raw[:, :-1]).pow(2).mean()


def bone_length_loss(pred_raw, target_raw):
    connections = [
        (11, 12), (11, 13), (13, 15),
        (12, 14), (14, 16),
        (11, 23), (12, 24),
        (23, 24),
        (23, 25), (25, 27),
        (24, 26), (26, 28),
    ]

    pred = pred_raw.reshape(pred_raw.shape[0], pred_raw.shape[1], 33, 2)
    target = target_raw.reshape(target_raw.shape[0], target_raw.shape[1], 33, 2)

    loss = 0.0
    for a, b in connections:
        pred_len = torch.norm(pred[:, :, a] - pred[:, :, b], dim=-1)
        target_len = torch.norm(target[:, :, a] - target[:, :, b], dim=-1)
        loss = loss + torch.mean((pred_len - target_len) ** 2)

    return loss / len(connections)


X = np.load(DATA_DIR / "X_csi.npy").astype(np.float32)
Y_raw = np.load(DATA_DIR / "Y_pose_seq.npy").astype(np.float32)
Y_norm = np.load(DATA_DIR / "Y_pose_seq_norm.npy").astype(np.float32)
visibility = np.load(DATA_DIR / "Y_visibility_seq.npy").astype(np.float32)
splits = np.load(DATA_DIR / "split_indices.npz")

train_idx = splits["train_idx"]
val_idx = splits["val_idx"]
test_idx = splits["test_idx"]

Y_raw = Y_raw.reshape(Y_raw.shape[0], Y_raw.shape[1], -1)
Y_norm = Y_norm.reshape(Y_norm.shape[0], Y_norm.shape[1], -1)

Y = np.concatenate([Y_raw, Y_norm], axis=-1)

pose_visibility = np.repeat(visibility[..., None], 2, axis=-1).reshape(Y_raw.shape)
target_visibility = np.concatenate(
    [pose_visibility, pose_visibility * AUX_WEIGHT],
    axis=-1
)

x_mean = X[train_idx].mean(axis=(0, 1), keepdims=True).astype(np.float32)
x_std = X[train_idx].std(axis=(0, 1), keepdims=True).astype(np.float32)
x_std = np.where(x_std < 1e-6, 1.0, x_std).astype(np.float32)

X = (X - x_mean) / x_std

train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X[train_idx], dtype=torch.float32),
        torch.tensor(Y[train_idx], dtype=torch.float32),
        torch.tensor(target_visibility[train_idx], dtype=torch.float32),
        torch.tensor(Y_raw[train_idx], dtype=torch.float32),
    ),
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    TensorDataset(
        torch.tensor(X[val_idx], dtype=torch.float32),
        torch.tensor(Y[val_idx], dtype=torch.float32),
        torch.tensor(target_visibility[val_idx], dtype=torch.float32),
        torch.tensor(Y_raw[val_idx], dtype=torch.float32),
    ),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    TensorDataset(
        torch.tensor(X[test_idx], dtype=torch.float32),
        torch.tensor(Y[test_idx], dtype=torch.float32),
        torch.tensor(target_visibility[test_idx], dtype=torch.float32),
        torch.tensor(Y_raw[test_idx], dtype=torch.float32),
    ),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

model = CSIPoseBiLSTMAttention(
    input_dim=X.shape[2],
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    output_dim=Y.shape[2],
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5,
)

best_val = float("inf")
best_epoch = 0


def run_epoch(loader, is_train):
    model.train(is_train)

    total_loss = 0
    total_pose_loss = 0
    total_items = 0

    for batch_x, batch_y, batch_visibility, batch_y_raw in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        batch_visibility = batch_visibility.to(device)
        batch_y_raw = batch_y_raw.to(device)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            pred = model(batch_x)

            pred_raw = pred[:, :, :66]

            pose_loss = weighted_mse(pred, batch_y, batch_visibility)
            smooth_loss = temporal_smoothness_loss(pred_raw)
            bone_loss = bone_length_loss(pred_raw, batch_y_raw)

            loss = pose_loss + 0.01 * smooth_loss + 0.02 * bone_loss

            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        bs = batch_x.shape[0]
        total_loss += loss.item() * bs
        total_pose_loss += pose_loss.item() * bs
        total_items += bs

    return total_loss / total_items, total_pose_loss / total_items


for epoch in range(1, EPOCHS + 1):
    train_loss, train_pose = run_epoch(train_loader, True)
    val_loss, val_pose = run_epoch(val_loader, False)

    scheduler.step(val_loss)

    print(
        f"epoch {epoch:03d} | "
        f"train={train_loss:.6f} | "
        f"val={val_loss:.6f} | "
        f"val_pose={val_pose:.6f}"
    )

    if val_loss < best_val:
        best_val = val_loss
        best_epoch = epoch

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "x_mean": x_mean,
                "x_std": x_std,
                "input_dim": X.shape[2],
                "output_dim": Y.shape[2],
                "hidden_dim": HIDDEN_DIM,
                "num_layers": NUM_LAYERS,
                "window_size": X.shape[1],
                "target": "raw_and_normalized_sequence",
                "best_val_loss": best_val,
                "best_epoch": best_epoch,
            },
            MODEL_PATH,
        )

        print("saved:", MODEL_PATH)

    if epoch - best_epoch >= PATIENCE:
        print("early stopping:", epoch)
        break


checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])

test_loss, test_pose = run_epoch(test_loader, False)

print("best_epoch:", checkpoint["best_epoch"])
print("best_val_loss:", checkpoint["best_val_loss"])
print("test_loss:", test_loss)
print("test_pose_loss:", test_pose)

In [ ]:
import matplotlib.pyplot as plt
import random
import pandas as pd

checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

info_df = pd.read_csv(DATA_DIR / "dataset_info.csv")

POSE_CONNECTIONS = [
    (11, 12), (11, 13), (13, 15),
    (12, 14), (14, 16),
    (11, 23), (12, 24),
    (23, 24),
    (23, 25), (25, 27),
    (24, 26), (26, 28),
]

def draw_pose(points, title):
    plt.figure(figsize=(5, 6))

    for a, b in POSE_CONNECTIONS:
        plt.plot(
            [points[a, 0], points[b, 0]],
            [points[a, 1], points[b, 1]],
        )

    plt.scatter(points[:, 0], points[:, 1])
    plt.gca().invert_yaxis()
    plt.xlim(0, 1)
    plt.ylim(1, 0)
    plt.grid(True)
    plt.title(title)
    plt.show()


idx = random.choice(test_idx.tolist())

x_sample = torch.tensor(X[idx:idx+1], dtype=torch.float32).to(device)

with torch.no_grad():
    pred = model(x_sample).cpu().numpy()[0]

pred_raw = pred[:, :66].reshape(10, 33, 2)
true_raw = Y_raw[idx].reshape(10, 33, 2)

print(info_df.loc[idx, ["sample_id", "action", "split", "start_time_sec", "end_time_sec"]])

draw_pose(true_raw[-1], "GT Skeleton")
draw_pose(pred_raw[-1], "Predicted Skeleton")